In [2]:
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv("estaciones.csv")
df_vagones = df[df["num_vag"]>3]
df_oper = df[df["eta_oper"]>1]

In [4]:
#GeodataFrame de todas las estaciones
gdf = gpd.GeoDataFrame(
    df,
    geometry = gpd.points_from_xy(
        df["longitud"],
        df["latitud"]
    ),
    crs = "EPSG:4326"
)

#GeodataFrame de las estaciones más grandes (>3 vagones)
gdf_vagones = gpd.GeoDataFrame(
    df_vagones,
    geometry = gpd.points_from_xy(
        df_vagones["longitud"],
        df_vagones["latitud"]
    ),
    crs = "EPSG:4326"
)

gdf_oper = gpd.GeoDataFrame(
    df_oper,
    geometry = gpd.points_from_xy(
        df_oper["longitud"],
        df_oper["latitud"]
    ),
    crs = "EPSG:4326"
)

In [5]:
filtro_gdf = gdf[["nom_est", "latitud", "longitud", "geometry", "id_trazado", "tipo_esta"]]
filtro_vagones = gdf_vagones[["nom_est", "latitud", "longitud", "geometry", "num_vag"]]
filtro_oper = gdf_oper[["nom_est", "latitud", "longitud", "geometry"]]

In [6]:
import folium

# Muestra un mapa interactivo con OpenStreetMap de fondo
map = filtro_gdf.explore(
    column="id_trazado",       # indicar que columna define los grupos
    cmap="tab20b",             # Paleta especial con 20 colores distintos
    categorical=True,          # Obliga a tratar los IDs como grupos separados (no como un degradado)
    legend=True,
    marker_kwds={"radius": 3},
    tooltip=["nom_est", "id_trazado"],
    name="Estaciones"
)

filtro_gdf.explore(
    m = map,
    column="tipo_esta",       
    cmap="turbo",             
    categorical=True,          
    legend=False,               # Muestra el recuadro con los colores y sus tipos   
    marker_kwds={"radius": 3},
    tooltip=["nom_est", "tipo_esta"],
    name="Tipo de Estación"
)

filtro_vagones.explore(
    m = map,                   # Coloca los puntos en el mismo mapa "map"
    color="blue",               
    marker_kwds={"radius": 5},
    tooltip=["nom_est", "num_vag"],
    name="Estaciones Grandes (>3 vagones)"
)

filtro_oper.explore(
    m = map,                   
    color="red",               
    marker_kwds={"radius": 5}, 
    tooltip=["nom_est"],
    name="Cerrado por Obras"
)

#Crea la leyenda
folium.LayerControl().add_to(map)
map